# Huấn luyện NER gia phả (PhoBERT / SikuBERT) trên Colab
**Tốn tiền (GPU Colab).** Chọn *Runtime → Change runtime type → GPU* rồi chạy lần lượt. Cuối cùng chạy ô **Ngắt runtime**.

Chuẩn bị trên Google Drive, thư mục `MyDrive/family_tree_ner/`:
- `train_ner.py` (từ `research/label_studio_pipeline/`)
- `vi_v1/` hoặc `han_v1/` chứa `train.conll`, `dev.conll`, `test.conll` (sinh bằng `export_conll.py`)

In [ ]:
# 1) Cấu hình — chỉ sửa ô này
MODEL = "vinai/phobert-base"      # Việt. Hán: "SIKU-BERT/sikubert". KHÔNG dùng phobert-base-v2 (AGPL)
DATA  = "vi_v1"                    # Hán: "han_v1"
OUT   = "out_vi"                   # Hán: "out_han"
EPOCHS, LR, BATCH = 10, 3e-5, 16
BASE = "/content/drive/MyDrive/family_tree_ner"

In [ ]:
# 2) Gắn Drive, cài thư viện, kiểm tra GPU
from google.colab import drive
drive.mount("/content/drive")
!pip -q install "transformers>=4.40" 
import torch; print("GPU:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
assert torch.cuda.is_available(), "Chưa bật GPU — dừng để khỏi huấn luyện chậm."
!ls {BASE} {BASE}/{DATA}

In [ ]:
# 3) Huấn luyện (log mỗi epoch; lưu model tốt nhất theo dev F1 vào Drive)
!python {BASE}/train_ner.py --model {MODEL} --data-dir {BASE}/{DATA} --output-dir {BASE}/{OUT} \
    --epochs {EPOCHS} --lr {LR} --batch-size {BATCH}

In [ ]:
# 4) Xem kết quả (dev/test, F1 từng nhãn, khoảng tin cậy)
import json
card = json.load(open(f"{BASE}/{OUT}/model_card.json"))
print("best epoch:", card["best_epoch"])
print("DEV :", card["dev"]["micro"])
print("TEST:", card.get("test", {}).get("micro"), card.get("test", {}).get("micro_f1_ci95_bootstrap"))
for k, v in card.get("test", {}).get("per_label", {}).items(): print(k, v)

In [ ]:
# 5) NGẮT RUNTIME để dừng tính tiền (model đã nằm trong Drive)
from google.colab import runtime
runtime.unassign()